In [ ]:
import pandas as pd
import numpy as np

In [ ]:
#import data
df_weather = pd.read_csv('NYC2023Weather.csv')

In [ ]:
#data preview
df_weather.head()

In [ ]:
df_weather.columns

In [ ]:
#trim unnecessary data and rename columns
rename_map = {'STATION': 'Station', 'NAME': 'Name', 'LATITUDE': 'Latitude', 'LONGITUDE': 'Longitude', 'DATE': 'Date', 'PRCP': 'Precipitation', 'SNOW': 'Snow', 'TAVG': 'Avg_Temp', 'TMAX': 'Max_Temp', 'TMIN': 'Min_Temp'}
col_keep = ['STATION', 'NAME', 'LATITUDE', 'LONGITUDE', 'DATE', 'PRCP', 'SNOW', 'TAVG', 'TMAX', 'TMIN']

df_weather_trim = df_weather[col_keep].rename(columns = rename_map)
df_weather_trim.head()

In [ ]:
#how many n/a per row?
df_weather_trim.isna().sum(axis=1)

In [ ]:
#see which cities have workable data (based on what is needed, not including what can be calculated)
df_weather_trim = df_weather_trim.dropna(subset = ['Station','Name','Latitude','Longitude','Date','Max_Temp','Min_Temp', 'Precipitation','Snow'])
good_cities = df_weather_trim['Name'].unique()
good_cities

In [ ]:
#how many total data points (stations)?
len(good_cities)

In [ ]:
#categorize snow and rainy days in new column
df_weather_trim['Rain?'] = np.where(df_weather_trim['Precipitation'] > 0, 'yes', 'no')
df_weather_trim['Snow?'] = np.where(df_weather_trim['Snow'] > 0, 'yes', 'no')

In [ ]:
#fix wonky indexing
df_weather_trim = df_weather_trim.reset_index(drop = True)
df_weather_trim

In [ ]:
#some days are missing average values, let's assume the mean of Max_Temp and Min_Temp is Avg_Temp for those days only
temp_means = (df_weather_trim['Max_Temp'] + df_weather_trim['Min_Temp']) / 2
df_weather_trim['Avg_Temp'] = df_weather_trim['Avg_Temp'].fillna(temp_means)
df_weather_trim

In [ ]:
#yay!
df_weather_trim.isna().sum(axis=1).sum()

In [ ]:
#make sure all dates are in proper format and convert to date time
df_weather_trim['Date'] = pd.to_datetime(df_weather_trim['Date'])
df_weather_trim.info()

In [ ]:
#let's check how different weather is across a random day in NYC
rand_day = df_weather_trim[df_weather_trim['Date'] == '2023-04-01']
temp_stat = rand_day[['Avg_Temp', 'Max_Temp', 'Min_Temp']].agg(['mean','std'])
temp_stat

In [ ]:
#check for inconsistencies
df_weather_trim.describe()

In [ ]:
#let's check min of 0 degrees F makes sense
df_weather_trim.loc[df_weather_trim['Min_Temp'] == df_weather_trim['Min_Temp'].min()]
#according to google, 2/4/23-2/5/23 had extreme low temperatures

In [ ]:
# split dataset by state
df_ny_w = df_weather_trim[df_weather_trim['Name'].str.contains('NY')]
df_nj_w = df_weather_trim[df_weather_trim['Name'].str.contains('NJ')]

In [ ]:
#check that nothing was left out
len(df_ny_w) + len(df_nj_w) == len(df_weather_trim)

In [ ]:
#save overall datasets (un-hashtag to save)
#df_ny_w.to_csv('CLEANED_NYWeather2023.csv', index = False)
#df_nj_w.to_csv('CLEANED_NJWeather2023.csv', index = False)

In [ ]:
#check for consistency
df_ny_w.groupby('Date')['Avg_Temp'].std().mean()

In [ ]:
#narrow down to manhattan
df_ny_manhattan = df_ny_w[df_ny_w['Name'].str.contains('World Trade | Central Park', case = False)]

In [ ]:
#check if two stations are necessary
df_ny_manhattan.groupby('Date')['Avg_Temp'].std().mean()

In [ ]:
#seems like world trade center is missing at least one day
df_ny_manhattan[df_ny_manhattan['Date'] == '2023-01-01']

In [ ]:
#target world trade center data
trade = df_ny_w[df_ny_w['Name'] == 'WORLD TRADE CENTER, NY US']

In [ ]:
#check for total missing days
ref23 = pd.date_range('2023', '2024', inclusive='left', freq='D')
dti23 = pd.to_datetime(trade['Date'])

out = ref23.difference(dti23)  # missing dates here
out

In [ ]:
#since mean difference is ~ 2 degrees, and Central Park to World Trade Center is only ~5 miles,
#fair to assume NY Central Park encapsulates Manhattan weather

#check for missing days in Central Park
park = df_ny_w[df_ny_w['Name'] == 'NY CITY CENTRAL PARK, NY US']
dti23_2 = pd.to_datetime(park['Date'])

out2 = ref23.difference(dti23_2)  # missing dates here
out2

In [ ]:
#no missing days, save data below
#park.to_csv('CLEANED_CentralPark_Weather2023.csv', index = False)